# 03 - Fisher Information and Cramer-Rao Bounds

This notebook evaluates Fisher information and Cramer-Rao lower bounds (CRLBs) for the statistical-moment parameters estimated in `02_generate_measurements.ipynb`.

The calculation uses the same truncated spectral model as the moment estimator. For a parameter vector

&emsp;$\boldsymbol{\theta} = (\mu,\sigma,\gamma_3,\gamma_4,\ldots)$,

the normalized scatter spectrum is approximated as

&emsp;$S_N(\omega;\boldsymbol{\theta}) = \sum_{n=0}^{N} \frac{(-i\omega)^n}{n!}m_n(\boldsymbol{\theta})$,

where $m_n$ are the corresponding raw moments. The sampled measurement model is

&emsp;$H_k = A\,S_N(\omega_k;\boldsymbol{\theta})+\eta_k$,

where $A$ accounts for the amplitude scaling introduced when each return is normalized to unit average power and $\eta_k$ is the spectral noise after division by the known interrogating spectrum.

The Fisher information matrix is calculated from the analytic derivatives of the spectral model with respect to the moment parameters. CRLBs are evaluated both with $A$ treated as known and with $A$ included as an additional
nuisance parameter.

For calculations using sixth-order fits from `02_generate_measurements.ipynb`, hyperskewness and hyperkurtosis are treated as nuisance parameters.

**Numerical conditioning:** As the scatter distribution becomes increasingly sub-resolution, higher-order moment directions contribute progressively less independent information to the finite-bandwidth spectrum, and the Fisher information matrix can become strongly ill-conditioned. The inverse FIM is therefore evaluated using a Moore–Penrose pseudoinverse with rcond=1e-14. The FIM condition number is stored alongside each CRLB so that numerically unreliable regions can be identified during subsequent analysis.

## Setup

This notebook should be runnable from the repository root. The reusable analysis modules are loaded from `src/`.

## Inputs

Measurement databases generated by `02_generate_measurements.ipynb`, loaded from `data/returns/`

## Outputs

CRLB-enriched databases are written to `data/crlb/`

> The full calculation is computationally expensive. Output files are written incrementally and can be resumed after interruption.

In [ ]:
import sys
import math
import numpy as np
import pandas as pd

from pathlib import Path

In [ ]:
ROOT        = Path.cwd()
SRC_DIR     = ROOT / "src"
RETURNS_DIR = ROOT / "data" / "returns"
CRLB_DIR    = ROOT / "data" / "crlb"

if not SRC_DIR.is_dir():
    raise FileNotFoundError("Project root not found. Run this notebook with the repository "
                            "root as the current working directory.")

if not RETURNS_DIR.is_dir():
    raise FileNotFoundError("Return databases not found. Run 02_generate_measurements.ipynb first.")

CRLB_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import gmm
import databases as dbl
import scatterers as sfl
import spectral_estimation as pfl

## Core Functions

### Filename and database helpers

In [ ]:
# gather the snr value from the file name
def infer_snr_from_filename(filename):
    strings             = filename.replace('.csv', '').split('_')
    for string in strings:
        if 'snr' in string:
            snr_val     = string.replace('snr', '')
            if snr_val == 'NA':
                return None
            else:
                return float(snr_val)
    raise ValueError("Unknown SNR in filename.")

# gather the polynomial fit order from the file name
def infer_order_from_filename(filename):
    strings             = filename.replace('.csv', '').split('_')
    for string in strings:
        if 'order' in string:
            return int(string.replace('order', ''))
    raise ValueError("Unknown order in filename.")

# gather the scatterer type from the file name
def infer_scatterer_from_filename(filename):
    if 'delta' in filename:
        return 'delta'
    elif 'gmm' in filename:
        return 'gmm'
    else:
        raise ValueError("Unknown scatterer type in filename.")

# gather the type of spectral interrogating wave used from the file name
def infer_base_type_from_filename(filename):
    strings             = filename.replace('.csv', '').split('_')
    for string in strings:
        if 'discrete' in string:
            return 'discrete'
        elif 'continuous' in string:
            return 'continuous'
    raise ValueError("Unknown base type in filename.")

# gather the signal length in units of inverse bandwidth
# multiply this by sample rate to get total number of samples in the signal
def infer_len_from_filename(filename):
    strings             = filename.replace('.csv', '').split('_')
    for string in strings:
        if 'len' in string:
            return int(string.replace('len', ''))
    raise ValueError("Unknown length in filename.")

### Moment parameterization

In [ ]:
# return moments as saved in database, up to order max_order
def moment_parameters_from_row(row, max_order):
    params = [float(row['mean']),
              float(row['std']),
              float(row['skew']),
              float(row['kurtosis']),
              float(row['hyperskewness']),
              float(row['hyperkurtosis'])]
    return np.asarray(params[:max_order], dtype=float)

# return central moments from moment parameters (mean, std, skewness, ...)
def central_moments_from_theta(theta, max_order):
    theta       = np.asarray(theta, dtype=float)
    std         = float(theta[1])
    c           = {0: 1.0, 1: 0.0}
    if max_order >= 2:
        c[2]    = std**2
    for n in range(3, max_order+1):
        gamma_n = float(theta[n-1])  # theta[2] = gamma_3
        c[n]    = gamma_n * std**n
    return c

# return raw moments from moment parameters (mean, std, skewness, ...)
def raw_moments_from_theta(theta, max_order):
    if len(theta) < max_order:
        raise ValueError(f'Too few parameters to calculate for max_order={max_order}.' +
                         f' number of parameters: {len(theta)}')
    theta       = np.asarray(theta, dtype=float)
    mean        = float(theta[0])
    c           = central_moments_from_theta(theta, max_order)
    raw         = np.zeros(max_order+1, dtype=float)
    for j in range(max_order+1):
        total   = 0.0
        for r in range(j+1):
            total += math.comb(j, r) * mean**(j-r) * c.get(r, 0.0)
        raw[j]  = total
    return raw

# Analytically evaluate derivatives of raw moments with respect to the moment parameters (mean, std, skew, ...)
# creates an N+1 x N matrix where J[j, a] = d m_j / d theta_a.
# rows are raw moment orders (including m_0), columns are reported params (starting at mean).
def raw_moment_jacobian(theta, max_order):
    theta           = np.asarray(theta, dtype=float)
    mean            = float(theta[0])
    std             = float(theta[1])
    c               = central_moments_from_theta(theta, max_order=max_order)
    J               = np.zeros((max_order + 1, max_order), dtype=float)
    for j in range(1, max_order+1):
        # d m_j / d mean
        d_mean      = 0.0
        for r in range(j):
            d_mean += (math.comb(j, r) * (j-r) * mean**(j-r-1) * c.get(r, 0.0))
        J[j, 0]     = d_mean
        # d m_j / d std
        d_std       = 0.0
        if j >= 2:
            d_std  += math.comb(j, 2) * mean**(j-2) * 2.0 * std
        for r in range(3, j+1):
            gamma_r = float(theta[r-1])
            d_std  += math.comb(j, r) * mean**(j-r) * gamma_r * r * std**(r-1)
        J[j, 1]     = d_std
        # d m_j / d gamma_p, p=3..N
        for p in range(3, max_order+1):
            if p <= j:
                J[j, p-1] = math.comb(j, p) * mean**(j-p) * std**p
    return J

### Spectral model and derivatives

In [ ]:
# calculate the spectrum from the truncated list of raw moments
def truncated_spectrum_from_raw_moments(omegas, raw_moments):
    omegas      = np.asarray(omegas, dtype=float)
    raw_moments = np.asarray(raw_moments, dtype=float)
    S           = np.zeros_like(omegas, dtype=complex)
    for n, coef in enumerate(raw_moments):
        S      += coef * ((-1j*omegas)**n / math.factorial(n))
    return S

# return the derivatives of each frequency with respect to each moment parameter in theta.
# returns a matrix with shape (len(omegas), max_order)
def spectrum_derivatives_theta(omegas, theta, max_order):
    omegas      = np.asarray(omegas, dtype=float)
    Jm          = raw_moment_jacobian(theta, max_order)
    K           = len(omegas)
    D           = np.zeros((K, max_order), dtype=complex)
    for n in range(1, max_order + 1):
        basis   = ((-1j*omegas)**n / math.factorial(n))[:, None]
        D      += basis * Jm[n, :][None, :]
    return D

### Noise model and Fisher information

In [ ]:
# returns a time series of the scatter distribution
def scatter_from_row(row, scatter_type, x):
    if scatter_type == 'delta':
        delta   = sfl.Delta(x, loc1=row["loc_1"], loc2=row["loc_2"], loc3=row["loc_3"],
                            amp1=row["amp_1"], amp2=row["amp_2"], amp3=row["amp_3"])
        return delta.scatter
    elif scatter_type == 'gmm':
        params  = [row["mu1"], row["mu2"], row["sigma"], row["w1"]]
        return gmm.gmm_pdf(x, params)
    else:
        raise ValueError(f"Unknown scatterer: {scatter_type}")

# finds the adjusted scatterer spectral amplitude at DC after normalizing the
# return signal for power in time. used for spectral SNR computation
def row_intercept_amplitude(row, scatter_type, dbconfig):
    scatterer       = scatter_from_row(row, scatter_type, dbconfig.t)
    return_sig      = dbconfig.sr.create_return_wave(scatterer, add_noise=False, real=dbconfig.real)
    R_coefs, _, _   = pfl.extract_return_coefs(return_sig, dbconfig.t, dbconfig.freqs, real=dbconfig.real)
    H_coefs         = R_coefs / dbconfig.G_fft_coefs
    A               = H_coefs[0]
    return(float(np.abs(A)))

# finds the variance for eta_k = N_k / G_k, where N is white noise at a known SNR
def eta_variance(dbconfig):
    if dbconfig.snr is None or (isinstance(dbconfig.snr, str) and str(dbconfig.snr).upper() == "NA"):
        return np.zeros(len(dbconfig.G_coefs), dtype=float)
    snr_linear      = 10.0**(float(dbconfig.snr)/10.0)
    fft_noise_var   = dbconfig.samples / snr_linear
    return fft_noise_var / np.abs(dbconfig.G_fft_coefs)**2

# calculates the crlb covariance matrix for theta: H_k = A * S_N(omega_k, theta) + eta_k,
# where A is the scalar multiplication factor to make the return signal normalized in time before adding noise.
# include_intercept=True assumes the actual intercept is not perfectly known. this will likely
# increase the CRLB, but accounts for uncertainty in the fitted intercept.
def crlb_covariance_theta(omegas, theta, eta_var, max_order, include_intercept=True, amplitude=1.0):
    omegas          = np.asarray(omegas, dtype=float)
    theta           = np.asarray(theta, dtype=float)
    eta_var         = np.asarray(eta_var, dtype=float)
    if np.any(eta_var <= 0):
        return np.zeros((len(theta), len(theta)), dtype=float), np.inf
    raw             = raw_moments_from_theta(theta, max_order)
    S               = truncated_spectrum_from_raw_moments(omegas, raw)
    D_theta         = amplitude * spectrum_derivatives_theta(omegas, theta, max_order)
    if include_intercept:
        D           = np.column_stack([D_theta, S])
    else:
        D           = D_theta
    weighted        = D / eta_var[:, None]
    fim             = 2.0 * np.real(D.conj().T @ weighted)
    cond            = np.linalg.cond(fim)
    cov_full        = np.linalg.pinv(fim, rcond=1e-14)
    cov_theta       = cov_full[:len(theta), :len(theta)]
    return cov_theta, cond

# return crlbs for each moment parameter in theta.
# returns variance bounds; take the square root when comparing them with estimator RMSE
def crlb_from_cov(cov_theta, max_order, include_intercept=True, fim_cond=None):
    suffix                                  = '_with_intercept' if include_intercept else ''
    out                                     = {}
    out[f'crlb_mean{suffix}']               = cov_theta[0, 0]
    out[f'crlb_std{suffix}']                = cov_theta[1, 1]
    if max_order >= 3:
        out[f'crlb_skew{suffix}']           = cov_theta[2, 2]
    if max_order >= 4:
        out[f'crlb_kurtosis{suffix}']       = cov_theta[3, 3]
    if max_order >= 5:
        out[f'crlb_hyperskewness{suffix}']  = cov_theta[4, 4]
    if max_order >= 6:
        out[f'crlb_hyperkurtosis{suffix}']  = cov_theta[5, 5]

    if fim_cond is not None:
        out[f'crlb_fim_cond{suffix}']   = fim_cond

    return out

def crlb_for_row(row, dbconfig, scatterer, eta_var, theta=None, amplitude=None, include_intercept=True):
    if theta is None:
        theta       = moment_parameters_from_row(row, dbconfig.max_fit_order)
    if amplitude is None:
        amplitude   = row_intercept_amplitude(row, scatterer, dbconfig)
    cov_theta, cond = crlb_covariance_theta(dbconfig.omegas, theta, eta_var, dbconfig.max_fit_order,
                                            include_intercept=include_intercept, amplitude=amplitude)
    out             = crlb_from_cov(cov_theta, dbconfig.max_fit_order, include_intercept=include_intercept,
                                    fim_cond=cond)
    return out

### Database processing

In [ ]:
def add_crlb_columns_to_database(input_file, output_dir, chunk_size=50000):
    input_file      = Path(input_file)
    output_dir      = Path(output_dir)
    filename        = input_file.name
    output_name     = f"{input_file.stem}_crlb.csv"
    output_file     = output_dir / output_name
    df_in           = pd.read_csv(input_file)
    dbconfig        = dbl.DB_Config()
    scatterer       = infer_scatterer_from_filename(filename)
    base_type       = infer_base_type_from_filename(filename)
    signallen       = infer_len_from_filename(filename)
    snr             = infer_snr_from_filename(filename)
    order           = infer_order_from_filename(filename)
    start           = 0
    expected_cols   = None

    dbconfig.edit_scatterer_type(scatterer)
    dbconfig.edit_base_type(base_type)
    dbconfig.make_interrogating_wave(SignalLen=signallen)
    dbconfig.add_noise_vals('NA' if snr is None else snr)
    dbconfig.set_testing_vars(max_fit_order=order)

    eta_var         = eta_variance(dbconfig)

    if output_file.exists():
        df_out      = pd.read_csv(output_file)
        start       = len(df_out)
        print(f'  Resuming from record {start}.')
    else:
        start       = 0
        print('  Starting from scratch.')
    for start_idx in range(start, len(df_in), chunk_size):
        end_idx     = min(start_idx + chunk_size, len(df_in))
        chunk       = df_in.iloc[start_idx:end_idx].copy()
        crlb_rows   = []
        print(f'    Processing records {start_idx} to {end_idx - 1}')
        for i, (_, row) in enumerate(chunk.iterrows()):
            row_out = {}
            theta   = moment_parameters_from_row(row, dbconfig.max_fit_order)
            amp     = row_intercept_amplitude(row, scatterer, dbconfig)
            row_out.update(crlb_for_row(row, dbconfig, scatterer, eta_var, theta, amp, include_intercept=False))
            row_out.update(crlb_for_row(row, dbconfig, scatterer, eta_var, theta, amp, include_intercept=True))
            if expected_cols is None:
                expected_cols = list(row_out.keys())
            if list(row_out.keys()) != expected_cols:
                raise ValueError(f"CRLB column mismatch at source row {start_idx + i}.\n"
                                 f"Expected columns: {expected_cols}\n"
                                 f"Got columns:      {list(row_out.keys())}")
            crlb_rows.append(row_out)
        crlb_df     = pd.DataFrame(crlb_rows, columns=expected_cols, index=chunk.index)
        out_chunk   = pd.concat([chunk, crlb_df], axis=1)
        if output_file.exists():
            out_chunk.to_csv(output_file, mode="a", header=False, index=False)
        else:
            out_chunk.to_csv(output_file, mode="w", header=True, index=False)
    print('  Processing complete!')
    return output_file

## Validate the CRLB calculation

Evaluate the Fisher information for one representative record before processing the full databases

In [ ]:
test_file = RETURNS_DIR / "delta_discrete_len20_snr20_order4.csv"

test_db = pd.read_csv(test_file, nrows=1)

dbconfig = dbl.DB_Config(
    base_type='discrete',
    scatterer='delta'
)

dbconfig.make_interrogating_wave()
dbconfig.add_noise_vals(20)
dbconfig.set_testing_vars(max_fit_order=4)

eta_var = eta_variance(dbconfig)

test_crlb = crlb_for_row(
    test_db.iloc[0],
    dbconfig,
    'delta',
    eta_var
)

pd.Series(test_crlb)

## Generate the full CRLB databases

Evaluates the Fisher information for every record in the discrete point-scatterer simulations used in the paper's estimator-performance comparisons, and writes the original measurement data together with the corresponding CRLB columns to `data/crlb/`. Noiseless databases are excluded.

The calculation is computationally expensive. Existing output files are resumed from their current number of records.

In [ ]:
return_files = sorted(RETURNS_DIR.glob("delta_discrete_*.csv"))

for input_file in return_files:
    if 'snrNA' in input_file.name:
        continue
    print(input_file.name)
    add_crlb_columns_to_database(input_file, CRLB_DIR)